In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# How to Compute the Conjunctive Normal Form

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [2]:
import { LogicParser } from './PropositionalLogicParser';
import { RecursiveSet, Tuple } from 'recursive-set';

In [3]:
type Variable = string;
type Formula  = Variable | ['⊤' | '⊥'] | ['¬', Formula] | ['↔' | '→' | '∧' | '∨', Formula, Formula];
type Literal  = Variable | Tuple<['¬', Variable]>;
type Clause   = RecursiveSet<Literal>;
type CNF      = RecursiveSet<Clause>;

In [ ]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

The function `eliminateBiconditional(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$

In [5]:
type Formula1  = Variable | ['⊤' | '⊥'] | ['¬', Formula1] | ['→' | '∧' | '∨', Formula1, Formula1];

In [6]:
function eliminateBiconditional(f: Formula): Formula1 {
    'Eliminate the logical operator "↔" from f.'
    if (typeof f === 'string') {
        return f;
    }
    const [op] = f;
    switch (op) {
        case '↔': {
            const [, g, h] = f;
            return eliminateBiconditional(['∧', ['→', g, h], ['→', h, g]]);
        }
        case '⊤':
            return ['⊤'];
        case '⊥':
            return ['⊥'];
        
        case '¬': {
            const [, g] = f;
            return ['¬', eliminateBiconditional(g)];
        }
        case '→':
        case '∧':
        case '∨': {
            const [, g, h] = f;
            return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
        }
    }
}

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from propositional logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$

In [7]:
type Formula2 = Variable | ['⊤' | '⊥'] | ['¬', Formula2] | ['∧' | '∨', Formula2, Formula2];

In [8]:
function eliminateConditional(f: Formula1): Formula2 {
    'Eliminate the logical operator "→" from f.'
    // variables.
    if (typeof f === 'string') { 
        return f; 
    }
    const [op] = f;
    switch (op) {
        case '⊤':
        case '⊥':
            return f;
        case '→': {
            const [, g, h] = f;
            return eliminateConditional(['∨', ['¬', g], h]);
        }
        case '¬': {
            const [, g] = f;
            return ['¬', eliminateConditional(g)];
        }
        case '∧':
        case '∨': {
            const [, g, h] = f;
            return [op, eliminateConditional(g), eliminateConditional(h)];
        }
    }
}

In [9]:
type NNF = Variable | ['⊤'] | ['⊥'] | ['¬', Variable] | ['∧' | '∨', NNF, NNF];

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

The forward declaration for the function `neg` is needed to typecheck the function `nnf`.

In [10]:
function nnf(f: Formula2): NNF {
    "Compute the negation normal form of f."
    if (typeof f === 'string') {
        return f;
    }
    const [op] = f;
    switch (op) {
        case '⊤':
            return ['⊤']
        case '⊥':
            return ['⊥'];
        case '¬': {
            const [, g] = f;
            return neg(g);
        }
        case '∧':
        case '∨': {
            const [, g, h] = f;
            return [op, nnf(g), nnf(h)];
        }
    }
}

function neg(f: Formula2): NNF {
    "Compute the negation normal form of ¬f."
    if (typeof f === 'string') {
        return ['¬', f];
    }
    const [op] = f;
    switch (op) {
        case '⊤':
            return ['⊥'];
        case '⊥':
            return ['⊤'];
        case '¬': {
            const [, g] = f;
            return nnf(g);
        }
        case '∧': {
            const [, g, h] = f;
            return ['∨', neg(g), neg(h)];
        }
        case '∨': {
            const [, g, h] = f;
            return ['∧', neg(g), neg(h)];
        }
    } 
}

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

In [11]:
const RS = RecursiveSet

In [12]:
function cnf(f: NNF): CNF {
  // f is a variable
  if (typeof f === 'string') {
    return new RS<Clause>(new RS<Literal>(f));
  }
  switch (f[0]) {
    case '⊤':
      return new RS<Clause>();
    case '⊥':
      return new RS<Clause>(new RS<Literal>());
    case '¬': {
      const [, p] = f;
      return new RS<Clause>(new RS<Literal>(new Tuple('¬', p)));
    }
    case '∧': {
      const [, g, h] = f;
      return cnf(g).union(cnf(h));
    }
    case '∨': {
      const [, g, h] = f;
      const res = new RS<Clause>();
      for (const c1 of cnf(g)) for (const c2 of cnf(h)) res.add(c1.union(c2));
      return res;
    }
  }
}


The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [13]:
function isTrivial(clause: Clause): boolean {
  return clause.some(
    lit => typeof lit === 'string' && clause.has(new Tuple('¬', lit))
  );
}

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [14]:
function simplify(clauses: CNF): CNF {
  return clauses.filter(c => !isTrivial(c));
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, trivial clausues are removed.

In [15]:
function normalize(f: Formula): CNF {
    const n1 = eliminateBiconditional(f);
    const n2 = eliminateConditional(n1);
    const n3 = nnf(n2);
    const n4 = cnf(n3);
    return simplify(n4);
}

In [16]:
function test(s: string): string {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    return normalize(f).toString();
}

In [17]:
test('(¬p → q) → (p → q) → q');

The knf of (¬p → q) → (p → q) → q is:
{}


In [18]:
test('(a → b) ↔ (¬a ∧ ¬b)');

The knf of (a → b) ↔ (¬a ∧ ¬b) is:
{{(¬, a), (¬, b)}, {(¬, b)}, {a, (¬, b)}}


In [19]:
test('(p ∧ q → r) ∨ ¬r → ¬p');

The knf of (p ∧ q → r) ∨ ¬r → ¬p is:
{{(¬, p), (¬, r)}, {q, (¬, p)}, {r, (¬, p)}}


In [20]:
test('⊤');

The knf of ⊤ is:
{}


In [21]:
test('⊥');

The knf of ⊥ is:
{{}}


In [22]:
test('(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p');

The knf of (p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p is:
{}


In [23]:
test("p → q");

The knf of p → q is:
{{q, (¬, p)}}


In [24]:
test("(p ∧ q) → r");

The knf of (p ∧ q) → r is:
{{r, (¬, p), (¬, q)}}


In [25]:
test("p ↔ q");

The knf of p ↔ q is:
{{p, (¬, q)}, {q, (¬, p)}}


In [26]:
test("(p → q) ∧ (q → r)");

The knf of (p → q) ∧ (q → r) is:
{{q, (¬, p)}, {r, (¬, q)}}


In [27]:
test("¬(p ∨ q)");

The knf of ¬(p ∨ q) is:
{{(¬, p)}, {(¬, q)}}


In [28]:
test('p ∧ p');

The knf of p ∧ p is:
{{p}}
